In [23]:

import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt

from tensorflow.keras.applications.efficientnet import preprocess_input

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
datadir = "/content/drive/MyDrive/Colab Notebooks/Data"
img_height = 224
img_weight =  224
batch_size = 16
seed = 42

In [26]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    datadir,
    validation_split = 0.2,
    subset = "training",
    image_size = (img_height, img_weight),
    batch_size = batch_size,
    seed = seed,
)

Found 872 files belonging to 6 classes.
Using 698 files for training.


In [27]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    datadir,
    validation_split = 0.2,
    subset = "validation",
    image_size = (img_height, img_weight),
    batch_size = batch_size,
    seed = seed,
)

Found 872 files belonging to 6 classes.
Using 174 files for validation.


In [28]:
class_names = train_dataset.class_names
num_classes = len(class_names)
print(class_names)

['Bird-drop', 'Clean', 'Dusty', 'Electrical-damage', 'Physical-Damage', 'Snow-Covered']


## Transfer Learning - EfficientNeV2B3

In [29]:
class_counts = {}
total_images = 0

for class_name in class_names:
  class_path = os.path.join(datadir, class_name)
  count = len(os.listdir(class_path))
  class_counts[class_name] = count
  total_images += count

print("Class Count: ", class_counts)

Class Count:  {'Bird-drop': 191, 'Clean': 194, 'Dusty': 192, 'Electrical-damage': 103, 'Physical-Damage': 69, 'Snow-Covered': 123}


In [30]:
class_weights = {}

for index, class_name in enumerate(class_names):
  class_weights[index] = total_images / (num_classes * class_counts[class_name])

print("Class weights: ", class_weights)

Class weights:  {0: 0.7609075043630017, 1: 0.7491408934707904, 2: 0.7569444444444444, 3: 1.4110032362459546, 4: 2.106280193236715, 5: 1.1815718157181572}


In [31]:
def prepare_data(dataset):
    images_list = []
    labels_list = []

    for imgs, lbls in dataset:
        # Assuming preprocess_input is already defined (e.g., from ResNet or MobileNet)
        processed_imgs = preprocess_input(imgs)
        images_list.append(processed_imgs)
        labels_list.append(lbls)

    # Concatenate into single large Tensors
    X = tf.concat(images_list, axis=0)
    y = tf.concat(labels_list, axis=0)

    # Flatten labels to 1D to ensure compatibility with class_weights
    y = tf.reshape(y, [-1])

    return X, y

In [32]:
train_images, train_labels = prepare_data(train_dataset)
val_images, val_labels = prepare_data(validation_dataset)

## Applying HPO on EfficientNetV2B3

In [34]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.layers import RandomFlip, RandomContrast,RandomRotation, RandomZoom, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import Sequential
from tensorflow.keras.optimizers import Adam

In [35]:
def build_model(hp):
    base_model = EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        include_preprocessing=True
    )

    base_model.trainable = False

    model = Sequential([
        RandomFlip("horizontal_and_vertical"),
        RandomRotation(0.05),
        RandomZoom(0.1),

        base_model,
        GlobalAveragePooling2D(),

        Dropout(0.4),
        Dense(256, activation='relu'),
        Dense(6, activation='softmax')  # your 6 classes
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [36]:


# Setup Random Search Tuner
tuner = kt.BayesianOptimization(
    build_model,
    objective='val_accuracy',
    max_trials=1,
    executions_per_trial=1,
    directory='kt_dir',
    project_name='efficientnet_bayesian_tune',
    overwrite=True
)

In [37]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

In [38]:



# Run the search
tuner.search(
    x=train_images,
    y=train_labels,
    validation_data=(val_images, val_labels),
    epochs=30,
    batch_size=16,
    callbacks=[early_stopping],
    class_weight=class_weights
)

# Results
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:")
print(best_hps.values)

best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate on validation
val_loss, val_acc = best_model.evaluate(validation_dataset)
print(f"Best validation accuracy: {val_acc:.4f}")

Trial 1 Complete [00h 33m 01s]
val_accuracy: 0.7988505959510803

Best val_accuracy So Far: 0.7988505959510803
Total elapsed time: 00h 33m 01s
Best hyperparameters:
{}


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


11/11 ━━━━━━━━━━━━━━━━━━━━ 17s 920ms/step - accuracy: 0.7989 - loss: 0.6444
Best validation accuracy: 0.7989


## Classification Report

In [39]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

# Get predictions from the best model on the validation set
val_predictions = best_model.predict(val_images)
predicted_classes = np.argmax(val_predictions, axis=1)

# Calculate F1-score
f1 = f1_score(val_labels, predicted_classes, average='weighted')
print(f"F1 Score: {f1:.4f}")

# Display classification report for more detailed metrics
print("\nClassification Report:")
print(classification_report(val_labels, predicted_classes, target_names=class_names))

6/6 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step
F1 Score: 0.7995

Classification Report:
                   precision    recall  f1-score   support

        Bird-drop       0.86      0.78      0.82        40
            Clean       0.64      0.91      0.75        33
            Dusty       0.86      0.79      0.83        48
Electrical-damage       0.73      0.85      0.79        13
  Physical-Damage       0.86      0.57      0.69        21
     Snow-Covered       0.94      0.89      0.92        19

         accuracy                           0.80       174
        macro avg       0.82      0.80      0.80       174
     weighted avg       0.82      0.80      0.80       174



In [40]:
best_model.save(f"/content/drive/MyDrive/Colab Notebooks/Model/effnetV2B0{val_acc:.2f}.h5")